# Proyecto — Data Stream Processor

## Contexto (extremadamente importante): 

Uno de las mayores virtudes de un repositorio en github es la poder volver sobre los cambios hechos. Uno puede volver sbre algún `commit` he iniciar el proceso desde ese punto. 

En otro ejemplo parecido, al crear una lista en python, dicha lista se modifica al agregar o borrar elementos y regresar a un estado anterior de la lista no es tan sencillo. La idea de este proyecto es poder emular dicho proceso y realizar una especie de lista con memoria para poder llevar algunos registros de manera adecuada.


### Objetivo

Construya un pequeño sistema para recibir y procesar registros de datos utilizando las clases `ArrayStack` y `ArrayQueue` proporcionadas por el curso. La idea central es poder llevar la información en orden para llevar los cambios de los registros de manera adecuada.

En el método `__init__` debe aparecer tres atributos:
- **Queue:** registros que han llegado pero todavía no han sido procesados.
- **Stack:** historial de cambios realizados, para poder deshacer los cambios más recientes.
- **Lista:** como se encuentran los registros actualmente

Las clases `ArrayStack` y `ArrayQueue` ya están implementadas. **No debe implementarlas nuevamente ni modificarlas.**

## 1. Registro de datos

Cada registro es una tupla de tres elementos:

```python
(sensor, variable, value)
```

Ejemplos:

```python
("S01", "temperature", 23.5)
("S02", "temperature", 25.1)
("S01", "humidity", 61.2)
```

Una combinación única `(sensor, variable)` identifica un dato dentro del estado actual.

## 2. Clase `DataProcessor`

Implemente:

```python
class DataProcessor:
    ...
```

Debe utilizar:

- un `ArrayQueue` para los registros pendientes;
- un `ArrayStack` para el historial de cambios;
- un `list` para mantener el estado actual.

### Restricción

No sustituya `ArrayQueue` o `ArrayStack` por `list`, `collections.deque` u otra estructura para realizar las funciones que corresponden a la Queue o al Stack.

La clase `DataProcessor` no debe imprimir resultados. Los métodos deben devolver los valores especificados. Las impresiones utilizadas para demostrar el funcionamiento deben realizarse en las celdas de prueba.

In [1]:
from goodrich.ch06.array_stack import ArrayStack
from goodrich.ch06.array_queue import ArrayQueue
from goodrich.exceptions import Empty

In [2]:
class DataProcessor:

    def __init__(self):
        # Creación de ArrayQueue (Almacena los registros pendientes)
        self.queue = ArrayQueue()  
        # Creación de ArrayStack (Almacena los datos requeridos para revertir (Actualizaciones))
        self.stack = ArrayStack()
        # Creación de lista (sensor, variable, valor) - (Estado actual de los registros almacenados)
        self.lista = []

    # Función que verifica que el registro tenga 3 componentes 
    # (sensor, variable, valor)
    def add(self, record):
        if len(record) != 3:
            raise ValueError("El registro debe tener exactamente tres componentes")

        sensor, variable, value = record

    # Condicional con levantamiento de excepción que verifica que el valor sea de tipo númerico
        if not isinstance(value, (int, float)):
            raise ValueError("El valor del registro debe ser numérico")
    # Agrega el registro cuando cumple con la condición a la cola sin procesarlo todavia
        self.queue.enqueue(record)

    # Función que retorna la cantidad de registros "pendientes de procesamiento en la cola"
    def pending(self):
        return len(self.queue)

    # Función que retira de la cola el siguiente registro
    # ArrayQueue garantiza que se procese la cola en orden (FIFO)
    def process_next(self):
        record = self.queue.dequeue()

        sensor, variable, value = record
    # Ciclo for para recorrer la lista "verificar la existencia de un dato igual (sensor, variable, valor)"
        for i in range(len(self.lista)):
            current_sensor, current_variable, current_value = self.lista[i]
            # Si el sensor y la variable existen se trata de una actualización de un dato (sobre escritura)
            if current_sensor == sensor and current_variable == variable:

            # Antes de modificar el valor, se almacena en el Stack el valor anterior (registro) 
            # para recuperlo mediante la función undo()
                self.stack.push(
                    (sensor, variable, current_value, True)
                )
            # Actualiza el valor del dato 
                self.lista[i] = (sensor, variable, value)

            # Retorna el registro procesasdo (Actualizado)
                return record

    # Si el sensor y la variable no existia anteriormente, se almacena en el Stack la información necesario 
    # para eliminar el registro si posteriormente se hace undo()
        self.stack.push(
            (sensor, variable, None, False)
        )
    # Agregar el nuevo registro al estado actual
        self.lista.append(record)
    # Retorna el registro procesado 
        return record

    # Función que recorre el estado actual del dato buscando el sensor y la variable solicitados
    def current_value(self, sensor, variable):
        for record in self.lista:
        # Si encuentra la combinación requerida retorna el valor actual
            if record[0] == sensor and record[1] == variable:
                return record[2]
        # Sí no encuentra la combinación requerida sensor y variable levanta la excepción KeyError
        raise KeyError((sensor, variable))

    # Función que toma del Stack el "último cambio", sí el Stack esta vacío, ArrayStack levanta
    # la excepción Empty
    def undo(self):
        sensor, variable, previous_value, existed = self.stack.pop()
        # Recorre el estado actual (registro que corresponde al cambio que se quiere deshacer)
        for i in range(len(self.lista)):
            current_sensor, current_variable, current_value = self.lista[i]
        # Verifica la existencia de sensor y variable en los registros
        # Si el dato existia antes del último cambio (Actualización) se restaurara
        # a su valor anterior
            if current_sensor == sensor and current_variable == variable:
                if existed:
        # Si por su parte el dato fue creado por el último registro (procesamiento) se 
        # elimina del estado actual
                    self.lista[i] = (sensor, variable, previous_value)
                else:
                    self.lista.pop(i)
                return



In [25]:
processor = DataProcessor()

print("Queue vacio:", processor.queue.is_empty())
print("Stack vacío:", processor.stack.is_empty())
print("Lista actual:", processor.lista)

Queue vacio: True
Stack vacío: True
Lista actual: []


## 3. `add(record)`

Agrega un registro a la `ArrayQueue`.

```python
processor.add(("S01", "temperature", 23.5))
```

Requisitos:

- agrega el registro a la cola;
- **no procesa** el registro;
- conserva el orden de llegada.

No se requiere un valor de retorno.

Debe rechazar registros que no tengan exactamente tres componentes o cuyo `value` no sea numérico. El tipo concreto de excepción para estos errores puede ser elegido por el estudiante, pero debe documentarse y utilizarse consistentemente.

In [26]:
processor = DataProcessor()

processor.add(("S01", "temperature", 23.5))

print("Registro en la Queue:", processor.queue.first())
print("Cantidad en la Queue:", len(processor.queue))
print("Estado actual:", processor.lista)

Registro en la Queue: ('S01', 'temperature', 23.5)
Cantidad en la Queue: 1
Estado actual: []


In [27]:
processor = DataProcessor()

processor.add(("S01", "temperature", 23.5))

print("Registros pendientes:", processor.pending())

Registros pendientes: 1


#Validación registro con menos de 3 componentes

In [6]:
processor = DataProcessor()

processor.add(("S01", "temperature"))

ValueError: El registro debe tener exactamente tres componentes

#Validación registro con más de 3 componentes

In [7]:
processor = DataProcessor()

processor.add(("S01", "temperature", 23.5, "extra"))

ValueError: El registro debe tener exactamente tres componentes

#Validación valor no númerico   

In [8]:
processor = DataProcessor()

processor.add(("S01", "temperature", "veintitrés"))

ValueError: El valor del registro debe ser numérico

## 4. `process_next()`

Procesa el siguiente registro pendiente.

Debe:

1. obtener el siguiente registro de la Queue;
2. procesarlo;
3. actualizar el estado actual;
4. guardar en el Stack la información necesaria para poder deshacer exactamente ese cambio.

### FIFO

Si se ejecuta:

```python
add(A)
add(B)
add(C)
```

las llamadas sucesivas a `process_next()` deben devolver/procesar `A`, luego `B` y luego `C`.

### Actualización

Si se procesa:

```python
("S01", "temperature", 23.5)
```

en el estado se debe reflejar:

```python
('S01', 'temperature', 23.5)
```

Si después se procesa `("S01", "temperature", 27.0)`, el valor actual debe ser `27.0`.

### Historial

Se debe agregar al `Stack` respectivo

### Retorno

Debe devolver el registro que acaba de ser procesado.

### Queue vacía

Si no hay registros pendientes, debe producir `Empty, el error creado en el repositorio `Goodrich`

In [30]:
processor = DataProcessor()
processor.add(("S01", "temperature", 23.5))

print("Pendientes antes:", processor.pending())

resultado = processor.process_next()

print("Registro procesado:", resultado)
print("Pendientes después:", processor.pending())
print("Estado actual:", processor.lista)

Pendientes antes: 1
Registro procesado: ('S01', 'temperature', 23.5)
Pendientes después: 0
Estado actual: [('S01', 'temperature', 23.5)]


In [ ]:
processor = DataProcessor()

#A - B tienen el mismo sensor - misma variable
A = ("S01", "temperature", 20)
B = ("S01", "temperature", 25)
C = ("S01", "humidity", 60)

# Agregar registros a la cola
processor.add(A)
processor.add(B)
processor.add(C)

print("Pendientes:", processor.pending())

Pendientes: 3


In [ ]:
# Agregar registro A
resultado_A = processor.process_next()

print("Procesado:", resultado_A)
print("Pendientes:", processor.pending())
print("Estado:", processor.lista)


Procesado: ('S01', 'temperature', 20)
Pendientes: 2
Estado: [('S01', 'temperature', 20)]


In [ ]:
# Agregar registro B
resultado_B = processor.process_next()

print("Procesado:", resultado_B)
print("Pendientes:", processor.pending())
print("Estado:", processor.lista)

Procesado: ('S01', 'temperature', 25)
Pendientes: 1
Estado: [('S01', 'temperature', 25)]


In [ ]:
# Agregar registro C
resultado_C = processor.process_next()

print("Procesado:", resultado_C)
print("Pendientes:", processor.pending())
print("Estado:", processor.lista)


Procesado: ('S01', 'humidity', 60)
Pendientes: 0
Estado: [('S01', 'temperature', 25), ('S01', 'humidity', 60)]


## 5. `undo()`

Deshace el último cambio realizado mediante `process_next()`.

Ejemplo:

```text
20 → 25 → 30
```

Después de un `undo()`:

```text
20 → 25
```

Después de otro:

```text
20
```

Los cambios deben deshacerse en orden LIFO.

### Dato creado por primera vez

Suponga que inicialmente no existe `('S01', 'temperature', x)`, después de procesar:

```python
("S01", "temperature", 23.5)
```

el dato existe. Si se ejecuta `undo()`, debe volver a **no existir**.

### Historial vacío

Si no hay cambios que deshacer, debe producir `Empty`.

No se requiere un valor de retorno.

In [14]:
processor = DataProcessor()

processor.add(("S01", "temperature", 20))
processor.process_next()

print("Antes de undo:", processor.lista)

processor.undo()

print("Después de undo:", processor.lista)

Antes de undo: [('S01', 'temperature', 20)]
Después de undo: []


In [15]:
processor = DataProcessor()

processor.add(("S01", "temperature", 20))
processor.process_next()

processor.add(("S01", "temperature", 25))
processor.process_next()

print("Antes de undo:", processor.lista)

processor.undo()

print("Después de undo:", processor.lista)

Antes de undo: [('S01', 'temperature', 25)]
Después de undo: [('S01', 'temperature', 20)]


In [16]:
processor = DataProcessor()

processor.add(("S01", "temperature", 20))
processor.add(("S01", "temperature", 25))
processor.add(("S01", "humidity", 60))

processor.process_next()
processor.process_next()
processor.process_next()

print("Estado antes de undo:", processor.lista)

processor.undo()
print("Después de undo 1:", processor.lista)

processor.undo()
print("Después de undo 2:", processor.lista)

processor.undo()
print("Después de undo 3:", processor.lista)

Estado antes de undo: [('S01', 'temperature', 25), ('S01', 'humidity', 60)]
Después de undo 1: [('S01', 'temperature', 25)]
Después de undo 2: [('S01', 'temperature', 20)]
Después de undo 3: []


In [17]:
processor = DataProcessor()

processor.undo()

Empty: Stack is empty

## 6. `pending()`

Devuelve el número de registros que todavía esperan ser procesados.

Por ejemplo, después de:

```python
add(A)
add(B)
add(C)
```

`pending()` debe devolver `3`. Después de `process_next()`, debe devolver `2`.

In [18]:
processor = DataProcessor()

print("Inicial:", processor.pending())

processor.add(("S01", "temperature", 20))
processor.add(("S01", "humidity", 60))

print("Después de agregar:", processor.pending())

processor.process_next()

print("Después de procesar uno:", processor.pending())

processor.process_next()

print("Después de procesar dos:", processor.pending())

Inicial: 0
Después de agregar: 2
Después de procesar uno: 1
Después de procesar dos: 0


## 7. `current_value(sensor, variable)`

Devuelve el valor actual asociado con una combinación de sensor y variable.

Ejemplo:

```python
current_value("S01", "temperature")
```

puede devolver `23.5`.

Si nunca se ha procesado un registro para esa combinación, debe producir `KeyError`.

In [19]:
processor = DataProcessor()

processor.add(("S01", "temperature", 23.5))
processor.add(("S01", "humidity", 61.2))

processor.process_next()
processor.process_next()

print("Temperatura:", processor.current_value("S01", "temperature"))
print("Humedad:", processor.current_value("S01", "humidity"))

Temperatura: 23.5
Humedad: 61.2


In [20]:
processor = DataProcessor()

processor.add(("S01", "temperature", 23.5))
processor.process_next()

processor.current_value("S02", "temperature")

KeyError: ('S02', 'temperature')

# 8. Ejemplo completo

Considere:

```python
A = ("S01", "temperature", 20)
B = ("S01", "temperature", 25)
C = ("S01", "humidity", 60)
```

Después de `add(A)`, `add(B)`, `add(C)`, la Queue contiene `A → B → C`.

Después de procesar A, el estado contiene:

```text
S01 / temperature → 20
```

Después de procesar B:

```text
S01 / temperature → 25
```

Después de procesar C:

```text
S01 / temperature → 25
S01 / humidity    → 60
```

Un `undo()` elimina el efecto de C. Otro `undo()` elimina el efecto de B. Otro `undo()` elimina el efecto de A y el estado vuelve a estar vacío.

# 9. Pruebas obligatorias

Incluya pruebas para, como mínimo:

1. `pending()` sobre un procesador vacío.
2. Agregar un registro.
3. Agregar varios registros.
4. Verificar procesamiento FIFO.
5. Procesar un registro.
6. Procesar varios registros.
7. Actualizar una variable existente.
8. Consultar el valor actual.
9. Realizar un `undo()`.
10. Realizar varios `undo()` consecutivos.
11. Procesar cuando la Queue está vacía.
12. Hacer `undo()` cuando el historial está vacío.
13. Deshacer la creación de un dato que antes no existía.
14. Hacer varios cambios sobre la misma variable.
15. Agregar un registro con formato incorrecto.
16. Agregar un registro cuyo valor no sea numérico.
17. Consultar un sensor/variable que nunca haya sido procesado.

##1. Pending sobre un procesador vacio

In [21]:
processor = DataProcessor()

print("Pendientes:", processor.pending())

Pendientes: 0


##2. Agregar un registro

In [22]:
processor = DataProcessor()

processor.add(("S01", "temperature", 23.5))

print("Pendientes:", processor.pending())
print("Estado actual:", processor.lista)

Pendientes: 1
Estado actual: []


##3. Agregar varios registros

In [23]:
processor = DataProcessor()

processor.add(("S01", "temperature", 23.5))
processor.add(("S02", "temperature", 25.1))
processor.add(("S03", "humidity", 61.2))

print("Pendientes:", processor.pending())
print("Primer registro:", processor.queue.first())
print("Estado actual:", processor.lista)

Pendientes: 3
Primer registro: ('S01', 'temperature', 23.5)
Estado actual: []


##4. Procesamiento FIFO

In [24]:
processor = DataProcessor()

processor.add(("S01", "temperature", 20))
processor.add(("S02", "temperature", 25))
processor.add(("S03", "humidity", 60))

print("1:", processor.process_next())
print("2:", processor.process_next())
print("3:", processor.process_next())

1: ('S01', 'temperature', 20)
2: ('S02', 'temperature', 25)
3: ('S03', 'humidity', 60)


##5. Procesar un registro   

In [28]:
processor = DataProcessor()

processor.add(("S01", "temperature", 23.5))

print("Antes:")
print("Pendientes:", processor.pending())
print("Estado:", processor.lista)

resultado = processor.process_next()

print("Después:")
print("Registro procesado:", resultado)
print("Pendientes:", processor.pending())
print("Estado:", processor.lista)

Antes:
Pendientes: 1
Estado: []
Después:
Registro procesado: ('S01', 'temperature', 23.5)
Pendientes: 0
Estado: [('S01', 'temperature', 23.5)]


##6. Procesar varios registros

In [29]:
processor = DataProcessor()

processor.add(("S01", "temperature", 20))
processor.add(("S02", "temperature", 25))
processor.add(("S03", "humidity", 60))

print("Pendientes iniciales:", processor.pending())

processor.process_next()
processor.process_next()
processor.process_next()

print("Pendientes finales:", processor.pending())
print("Estado actual:", processor.lista)

Pendientes iniciales: 3
Pendientes finales: 0
Estado actual: [('S01', 'temperature', 20), ('S02', 'temperature', 25), ('S03', 'humidity', 60)]


##7. Actualizar una variable existente

In [30]:
processor = DataProcessor()

processor.add(("S01", "temperature", 20))
processor.process_next()

print("Antes de actualizar:", processor.lista)

processor.add(("S01", "temperature", 25))
processor.process_next()

print("Después de actualizar:", processor.lista)

Antes de actualizar: [('S01', 'temperature', 20)]
Después de actualizar: [('S01', 'temperature', 25)]


##8 Consultar el valor actual

In [4]:
processor = DataProcessor()

processor.add(("S01", "temperature", 23.5))
processor.process_next()

print("Valor actual:", processor.current_value("S01", "temperature"))

Valor actual: 23.5


##9. Realizar un Undo()

In [5]:
processor = DataProcessor()

processor.add(("S01", "temperature", 23.5))
processor.process_next()

print("Antes de undo:", processor.lista)

processor.undo()

print("Después de undo:", processor.lista)

Antes de undo: [('S01', 'temperature', 23.5)]
Después de undo: []


##11. Realizar varios undo() consecutivos

In [6]:
processor = DataProcessor()

processor.add(("S01", "temperature", 20))
processor.process_next()

processor.add(("S01", "temperature", 25))
processor.process_next()

processor.add(("S01", "humidity", 60))
processor.process_next()

print("Antes de undo:", processor.lista)

processor.undo()
print("Después de undo 1:", processor.lista)

processor.undo()
print("Después de undo 2:", processor.lista)

processor.undo()
print("Después de undo 3:", processor.lista)

Antes de undo: [('S01', 'temperature', 25), ('S01', 'humidity', 60)]
Después de undo 1: [('S01', 'temperature', 25)]
Después de undo 2: [('S01', 'temperature', 20)]
Después de undo 3: []


##11. Procesar cuando la Queue está vacía

In [7]:
processor = DataProcessor()

processor.process_next()

Empty: Queue is empty

##12. Undo() cuando el historial está vacío

In [8]:
processor = DataProcessor()

processor.undo()

Empty: Stack is empty

##13. Deshacer la creación de un dato que antes no existía

In [9]:
processor = DataProcessor()

processor.add(("S01", "humidity", 60))
processor.process_next()

print("Después de procesar:", processor.lista)

processor.undo()

print("Después de undo:", processor.lista)

Después de procesar: [('S01', 'humidity', 60)]
Después de undo: []


##14. Varios cambios sobre la misma variable

In [10]:
processor = DataProcessor()

processor.add(("S01", "temperature", 20))
processor.process_next()

processor.add(("S01", "temperature", 25))
processor.process_next()

processor.add(("S01", "temperature", 30))
processor.process_next()

print("Estado actual:", processor.lista)

processor.undo()
print("Después de undo 1:", processor.lista)

processor.undo()
print("Después de undo 2:", processor.lista)

processor.undo()
print("Después de undo 3:", processor.lista)

Estado actual: [('S01', 'temperature', 30)]
Después de undo 1: [('S01', 'temperature', 25)]
Después de undo 2: [('S01', 'temperature', 20)]
Después de undo 3: []


##15. Registro con formato incorrecto

In [11]:
processor = DataProcessor()

processor.add(("S01", "temperature"))

ValueError: El registro debe tener exactamente tres componentes

##16. Valor no numérico

In [12]:
processor = DataProcessor()

processor.add(("S01", "temperature", "veintitrés"))

ValueError: El valor del registro debe ser numérico

##17. Consultar un sensor/variable nunca procesado

In [13]:
processor = DataProcessor()

processor.add(("S01", "temperature", 23.5))
processor.process_next()

processor.current_value("S02", "temperature")

KeyError: ('S02', 'temperature')

In [55]:

processor = DataProcessor()

processor.add(("S01", "temperature", 20))
processor.add(("S02", "temperature", 25))
processor.add(("S03", "humidity", 60))

print("Procesado 1:", processor.process_next())
print("Procesado 2:", processor.process_next())
print("Procesado 3:", processor.process_next())

Procesado 1: ('S01', 'temperature', 20)
Procesado 2: ('S02', 'temperature', 25)
Procesado 3: ('S03', 'humidity', 60)


# 10. Análisis de complejidad

Explique la complejidad temporal de:

- `add`
- `process_next`
- `undo`
- `pending`
- `current_value`

Justifique qué operaciones determinan cada complejidad e indique qué estructuras auxiliares utiliza el sistema.

Método add(record)
Complejidad O(1)
Razón:
La función valida el tamaño y el tipo de valor por lo que las comprobaciones tienen un número determinado de operaciones (enqueue() agrega el registro a la cola) por lo que su tiempo de ejecución es constante.


**def add(self, record):**
    **if len(record) != 3:**
        ...
    
    **sensor, variable, value = record**

    **if not isinstance(value, (int, float)):**
        ...

    **self.queue.enqueue(record)**

Método pending()
Complejidad O(1)
Razón:
La función no reccore ningún elemento por lo que su tiempo de ejecución es constante.

**def pending(self):**
    **return len(self.queue)**

Método process_next()
Complejidad O(n)

#Inicio se ejecuta la función con un tiempo de ejecución O(n)

**record = self.queue.dequeue()**

#Final se ejecuta el for para encontrar el registro correspondiente (sensor, variable), en el peor caso debe recorrer toda la lista por lo que su tiempo de ejecución es lineal.

**for i in range(len(self.lista)):**

Método current_value()
Complejidad O(n)

#Inicio se ejecuta la función con un tiempo de ejecución O(1)

**for record in self.lista:**

#Final se ejecuta el for para encontrar el registro correspondiente (sensor, variable), en el peor caso debe recorrer toda la lista por lo que su tiempo de ejecución es lineal.


Método undo()
Complejidad O(n)

#Inicio se ejecuta la función con un tiempo de ejecución O(1)

**self.stack.pop()**

#Final se ejecuta el for para encontrar el registro correspondiente, en el peor caso debe recorrer toda la lista, además cuando se elimina un elemento mediante (self.lista.pop(i)) 

**for i in range(len(self.lista)):**



# 11. Restricciones

1. Utilice las clases `ArrayStack` y `ArrayQueue` proporcionadas.
2. No las reemplace por `list`, `deque` u otra estructura equivalente.
3. No modifique las implementaciones proporcionadas.
4. La implementación debe estar contenida en `DataProcessor`.
5. Incluya las pruebas solicitadas.
6. Explique brevemente sus decisiones de diseño.

# 12. Bonus — `redo()`

Como extensión opcional, implemente:

```python
redo()
```

Después de un `undo()`, el sistema debe poder volver a aplicar el cambio que acaba de deshacerse.

Por ejemplo:

```text
20 → 25 → 30
undo()  → 20 → 25
redo()  → 20 → 25 → 30
```

El estudiante debe explicar qué estructuras utiliza para implementar `redo()` y por qué. No se proporciona la estrategia de implementación.


# 13. Entrega

La entrega debe contener:

- implementación completa de `DataProcessor`;
- pruebas solicitadas;
- explicación breve de las decisiones de diseño;
- análisis de complejidad temporal;
- si realiza el bonus, implementación y explicación de `redo()`.